# 01 — Dataset Assessment
**CareFlow Process Mining | Google Colab**

This notebook is tailored to `Sepsis_Cases_Event_Log_Cleaned_for_Visualization.xlsx`.
It uses the workbook's **Event Log Visual** sheet and its actual fields: `case_id`, `event_index`, `activity`, `timestamp`, `org_group`, and `lifecycle_transition`.

In [ ]:
!pip -q install openpyxl pandas numpy matplotlib seaborn pm4py

import io, os, re, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
print("CareFlow Process Mining environment ready.")

In [ ]:
# Upload: Sepsis_Cases_Event_Log_Cleaned_for_Visualization.xlsx
uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload the processed sepsis workbook.")
workbook_name = next(iter(uploaded))
workbook_bytes = uploaded[workbook_name]
xls = pd.ExcelFile(io.BytesIO(workbook_bytes))
print("Workbook:", workbook_name)
print("Sheets:", xls.sheet_names)

required_sheet = "Event Log Visual"
if required_sheet not in xls.sheet_names:
    raise ValueError(f"Expected sheet '{required_sheet}'. Available sheets: {xls.sheet_names}")

events = pd.read_excel(io.BytesIO(workbook_bytes), sheet_name="Event Log Visual")
events.columns = [str(c).strip() for c in events.columns]
print("Event Log Visual shape:", events.shape)
display(events.head())

In [ ]:
# This workbook's event-log sheet uses these verified column names:
CASE_COL = "case_id"
ACTIVITY_COL = "activity"
TIME_COL = "timestamp"
EVENT_INDEX_COL = "event_index"
ORG_COL = "org_group"
LIFECYCLE_COL = "lifecycle_transition"

# Parse timestamp safely; invalid values become NaT and are reported.
if TIME_COL in events.columns:
    events[TIME_COL] = pd.to_datetime(events[TIME_COL], errors="coerce")

print("Case column:", CASE_COL, "| present:", CASE_COL in events.columns)
print("Activity column:", ACTIVITY_COL, "| present:", ACTIVITY_COL in events.columns)
print("Timestamp column:", TIME_COL, "| present:", TIME_COL in events.columns)

## 1. Workbook and event-log overview

In [ ]:
print("Events (rows):", len(events))
print("Columns:", len(events.columns))
print("Unique cases:", events[CASE_COL].nunique(dropna=True))
print("Unique activities:", events[ACTIVITY_COL].nunique(dropna=True))
display(events.dtypes.astype(str).rename("data_type").to_frame())
display(events.head(10))

## 2. Data quality profile

In [ ]:
profile = pd.DataFrame({
    "column": events.columns,
    "data_type": [str(events[c].dtype) for c in events.columns],
    "non_null_count": [int(events[c].notna().sum()) for c in events.columns],
    "missing_count": [int(events[c].isna().sum()) for c in events.columns],
    "missing_percent": [(events[c].isna().mean()*100).round(2) for c in events.columns],
    "unique_values": [int(events[c].nunique(dropna=True)) for c in events.columns]
})
display(profile)
print("Exact duplicate rows:", int(events.duplicated().sum()))
print("Missing case IDs:", int(events[CASE_COL].isna().sum()))
print("Missing activities:", int(events[ACTIVITY_COL].isna().sum()))
print("Unparseable/missing timestamps:", int(events[TIME_COL].isna().sum()))

## 3. Event-log readiness checks

In [ ]:
print("Timestamp range:", events[TIME_COL].min(), "to", events[TIME_COL].max())
print("Cases with more than one event:", int((events.groupby(CASE_COL).size() > 1).sum()))
print("Lifecycle values:")
display(events[LIFECYCLE_COL].value_counts(dropna=False).to_frame("count"))
print("Organization groups:")
display(events[ORG_COL].value_counts(dropna=False).head(20).to_frame("count"))

## 4. Export reports

In [ ]:
profile.to_csv("dataset_profile.csv", index=False)
events.isna().sum().rename("missing_count").to_csv("missing_value_report.csv")
files.download("dataset_profile.csv")
files.download("missing_value_report.csv")